# GitHub Repository Auditor

This AI Agent audits Github repos in real time. Given a repository's owner and name, the agent will inspect structure, analyze dependencies, detect known CVEs and make a report with a list of priorized actions to do.

## Import Libraries

In [36]:
import requests
import json
import os
import base64
from typing import List, Optional
from dotenv import load_dotenv
import openai

In [37]:
load_dotenv()

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
GITHUB_API = "https://api.github.com"
OSV_API = "https://api.osv.dev/v1"

GITHUB_HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}

## Tool Functions

1. `get_repo_info`: repository's general metadata
2. `get_file_tree`: file directory complete structure
3. `get_file_content`: specific file contents
4. `get_dependencies`: project dependencies by language
5. `check_vulnerabilities`: known CVEs via OSV.dev

In [38]:
def get_repo_info(owner: str, repo: str) -> str:
    """
    Fetch general metadata about a GitHub repository.

    Args:
        owner: Repository owner (user or organization)
        repo: Repository name

    Returns:
        JSON string with repo metadata
    """
    url = f"{GITHUB_API}/repos/{owner}/{repo}"
    response = requests.get(url, headers=GITHUB_HEADERS)
    response.raise_for_status()
    data = response.json()

    return json.dumps({
        "name": data["full_name"],
        "description": data.get("description"),
        "language": data.get("language"),
        "languages_url": data.get("languages_url"),
        "license": data["license"]["name"] if data.get("license") else None,
        "stars": data["stargazers_count"],
        "forks": data["forks_count"],
        "open_issues": data["open_issues_count"],
        "default_branch": data["default_branch"],
        "created_at": data["created_at"][:10],
        "last_push": data["pushed_at"][:10],
        "size_kb": data["size"],
        "has_wiki": data["has_wiki"],
        "has_issues": data["has_issues"],
        "topics": data.get("topics", [])
    }, indent=2)

In [39]:
get_repo_info("psf", "requests")

'{\n  "name": "psf/requests",\n  "description": "A simple, yet elegant, HTTP library.",\n  "language": "Python",\n  "languages_url": "https://api.github.com/repos/psf/requests/languages",\n  "license": "Apache License 2.0",\n  "stars": 53895,\n  "forks": 9795,\n  "open_issues": 300,\n  "default_branch": "main",\n  "created_at": "2011-02-13",\n  "last_push": "2026-03-23",\n  "size_kb": 13331,\n  "has_wiki": true,\n  "has_issues": true,\n  "topics": [\n    "client",\n    "cookies",\n    "forhumans",\n    "http",\n    "humans",\n    "python",\n    "python-requests",\n    "requests"\n  ]\n}'

In [64]:
def get_file_tree(owner: str, repo: str, max_files: int = 100) -> str:
    """
    Fetch the full file/directory tree of a repository.

    Args:
        owner: Repository owner
        repo: Repository name
        max_files: Maximum number of file paths to return (default: 100)

    Returns:
        JSON string with file tree and structural analysis
    """
    url = f"{GITHUB_API}/repos/{owner}/{repo}/git/trees/HEAD"
    response = requests.get(url, headers=GITHUB_HEADERS, params={"recursive": "1"})
    response.raise_for_status()
    data = response.json()

    all_paths = [item["path"] for item in data.get("tree", [])]
    truncated = data.get("truncated", False)

    # Structural signals useful for the agent to reason about
    has_tests = any(
        p.startswith(("test", "tests", "__tests__", "spec", "specs"))
        or "/test" in p or "_test." in p or ".test." in p or ".spec." in p
        for p in all_paths
    )
    has_ci = any(
        p.startswith(".github/workflows") or p in (".travis.yml", "Jenkinsfile", ".circleci/config.yml")
        for p in all_paths
    )
    has_docker = any(p in ("Dockerfile", "docker-compose.yml", "docker-compose.yaml") for p in all_paths)
    has_readme = any(p.lower().startswith("readme") for p in all_paths)
    has_contributing = any("contributing" in p.lower() for p in all_paths)
    has_env_example = any(".env.example" in p or ".env.sample" in p for p in all_paths)
    has_gitignore = ".gitignore" in all_paths

    dependency_file_names = (
        "requirements.txt", "requirements-dev.txt", "Pipfile", "pyproject.toml",
        "package.json", "package-lock.json", "yarn.lock",
        "pom.xml", "Gemfile"
    )

    dependency_files = [
        p for p in all_paths
        if any(dep_name in p for dep_name in dependency_file_names)
    ]

    return json.dumps({
        "total_files": len(all_paths),
        "truncated": truncated,
        "paths": all_paths[:max_files],
        "signals": {
            "has_tests": has_tests,
            "has_ci": has_ci,
            "has_docker": has_docker,
            "has_readme": has_readme,
            "has_contributing_guide": has_contributing,
            "has_env_example": has_env_example,
            "has_gitignore": has_gitignore,
            "dependency_files_found": dependency_files
        }
    }, indent=2)

In [65]:
get_file_tree("sparisca05", "evagent")

'{\n  "total_files": 41,\n  "truncated": false,\n  "paths": [\n    ".gitignore",\n    ".vscode",\n    ".vscode/tasks.json",\n    "LICENSE",\n    "README.md",\n    "back",\n    "back/.gitignore",\n    "back/azure_ai_agent.py",\n    "back/main.py",\n    "back/requirements.txt",\n    "back/test",\n    "back/test/azure_ai_agent_test.py",\n    "front",\n    "front/.gitignore",\n    "front/README.md",\n    "front/eslint.config.js",\n    "front/index.html",\n    "front/package-lock.json",\n    "front/package.json",\n    "front/public",\n    "front/public/vite.svg",\n    "front/src",\n    "front/src/App.css",\n    "front/src/App.tsx",\n    "front/src/components",\n    "front/src/components/Chat",\n    "front/src/components/Chat/Chat.css",\n    "front/src/components/Chat/Chat.tsx",\n    "front/src/components/ConnectionSetup",\n    "front/src/components/ConnectionSetup/ConnectionSetup.css",\n    "front/src/components/ConnectionSetup/ConnectionSetup.tsx",\n    "front/src/components/Message",\n   

In [42]:
def get_file_content(owner: str, repo: str, path: str) -> str:
    """
    Fetch the content of a specific file in the repository.

    Args:
        owner: Repository owner
        repo: Repository name
        path: File path relative to repo root (e.g. 'README.md', 'src/main.py')

    Returns:
        File content as plain text (truncated to 8000 chars if large)
    """
    url = f"{GITHUB_API}/repos/{owner}/{repo}/contents/{path}"
    response = requests.get(url, headers=GITHUB_HEADERS)

    if response.status_code == 404:
        return f"File not found: {path}"

    response.raise_for_status()
    data = response.json()

    if data.get("encoding") == "base64":
        content = base64.b64decode(data["content"]).decode("utf-8", errors="replace")
    else:
        content = data.get("content", "")

    if len(content) > 8000:
        content = content[:8000] + f"\n\n[... truncated — full file is {len(content)} chars]"

    return content

In [44]:
get_file_content("psf", "requests", "src/requests/api.py")

'"""\nrequests.api\n~~~~~~~~~~~~\n\nThis module implements the Requests API.\n\n:copyright: (c) 2012 by Kenneth Reitz.\n:license: Apache2, see LICENSE for more details.\n"""\n\nfrom . import sessions\n\n\ndef request(method, url, **kwargs):\n    """Constructs and sends a :class:`Request <Request>`.\n\n    :param method: method for the new :class:`Request` object: ``GET``, ``OPTIONS``, ``HEAD``, ``POST``, ``PUT``, ``PATCH``, or ``DELETE``.\n    :param url: URL for the new :class:`Request` object.\n    :param params: (optional) Dictionary, list of tuples or bytes to send\n        in the query string for the :class:`Request`.\n    :param data: (optional) Dictionary, list of tuples, bytes, or file-like\n        object to send in the body of the :class:`Request`.\n    :param json: (optional) A JSON serializable Python object to send in the body of the :class:`Request`.\n    :param headers: (optional) Dictionary of HTTP Headers to send with the :class:`Request`.\n    :param cookies: (optiona

In [81]:
def get_dependencies(owner: str, repo: str) -> str:
    """
    Extract project dependencies by reading dependency files found in the repository tree.
    Supports Python (requirements.txt, requirements-dev.txt, pyproject.toml),
    Node.js (package.json), and Java (pom.xml).

    Args:
        owner: Repository owner
        repo: Repository name

    Returns:
        JSON string with detected ecosystems and list of dependencies with versions
    """
    file_tree = json.loads(get_file_tree(owner, repo))
    dependency_files = file_tree.get("signals", {}).get("dependency_files_found", [])
    ecosystems = {}

    for filepath in dependency_files:
        content = get_file_content(owner, repo, filepath)
        if content.startswith("File not found"):
            continue

        if "requirements.txt" in filepath or "requirements-dev.txt" in filepath:
            ecosystems.setdefault("python", {"source_file": filepath, "dependencies": []})
            for line in content.splitlines():
                line = line.strip()
                if line and not line.startswith("#") and not line.startswith("-"):
                    # Normalize: 'requests==2.28.0' or 'requests>=2.0'
                    for sep in ("==", ">=", "<=", "~=", "!=", ">", "<", "["):
                        if sep in line:
                            name, version = line.split(sep, 1)[0].strip(), line.split(sep, 1)[1].strip().split(";")[0]
                            ecosystems["python"]["dependencies"].append({"name": name, "version": version.split(",")[0]})
                            break
                    else:
                        ecosystems["python"]["dependencies"].append({"name": line, "version": None})

        elif "package.json" in filepath:
            ecosystems.setdefault("npm", {"source_file": filepath, "dependencies": []})
            try:
                pkg = json.loads(content)
                for name, version in {**pkg.get("dependencies", {}), **pkg.get("devDependencies", {})}.items():
                    ecosystems["npm"]["dependencies"].append({"name": name, "version": version.lstrip("^~><=!")})
            except json.JSONDecodeError:
                return f"Could not parse {filepath}"

        elif "pyproject.toml" in filepath:
            ecosystems.setdefault("python", {"source_file": filepath, "dependencies": []})
            # Basic extraction — looks for lines under [tool.poetry.dependencies] or [project]
            in_deps = False
            for line in content.splitlines():
                if "[tool.poetry.dependencies]" in line or "[project]" in line:
                    in_deps = True
                    continue
                if in_deps and line.startswith("["):
                    in_deps = False
                if in_deps and "=" in line and not line.startswith("#"):
                    name, version = line.split("=", 1)
                    ecosystems["python"]["dependencies"].append({"name": name.strip(), "version": version.strip().strip('"\' ')})

        elif "pom.xml" in filepath:
            ecosystems.setdefault("maven", {"source_file": filepath, "dependencies": []})
            # Basic extraction of dependencies from pom.xml
            in_deps = False
            current_dep = {}
            for line in content.splitlines():
                if "<dependencies>" in line:
                    in_deps = True
                    continue
                if "</dependencies>" in line:
                    in_deps = False
                if in_deps and "<dependency>" in line:
                    current_dep = {}
                if in_deps and "</dependency>" in line and current_dep.get("name"):
                    ecosystems["maven"]["dependencies"].append(current_dep)
                if in_deps and "<groupId>" in line:
                    current_dep["group"] = line.strip().replace("<groupId>", "").replace("</groupId>", "")
                if in_deps and "<artifactId>" in line:
                    current_dep["name"] = line.strip().replace("<artifactId>", "").replace("</artifactId>", "")
                if in_deps and "<version>" in line:
                    current_dep["version"] = line.strip().replace("<version>", "").replace("</version>", "")

    if ecosystems:
        return json.dumps({
            "ecosystems": [
                {
                    "ecosystem": ecosystem,
                    "source_file": data["source_file"],
                    "total": len(data["dependencies"]),
                    "dependencies": data["dependencies"]
                }
                for ecosystem, data in ecosystems.items()
            ]
        }, indent=2)

    return json.dumps({"error": "No supported dependency file found (requirements.txt, package.json, pyproject.toml, pom.xml)"})

In [82]:
get_dependencies("sparisca05", "evagent")

'{\n  "ecosystems": [\n    {\n      "ecosystem": "python",\n      "source_file": "back/requirements.txt",\n      "total": 12,\n      "dependencies": [\n        {\n          "name": "python-dotenv",\n          "version": "1.0.1"\n        },\n        {\n          "name": "fastapi",\n          "version": null\n        },\n        {\n          "name": "uvicorn",\n          "version": "0.22.0"\n        },\n        {\n          "name": "python-multipart",\n          "version": null\n        },\n        {\n          "name": "apify-client",\n          "version": null\n        },\n        {\n          "name": "pydantic",\n          "version": null\n        },\n        {\n          "name": "pandas",\n          "version": null\n        },\n        {\n          "name": "azure-search-documents",\n          "version": "11.5.2"\n        },\n        {\n          "name": "azure-ai-projects",\n          "version": null\n        },\n        {\n          "name": "openai",\n          "version": null\n     

In [ ]:
def check_vulnerabilities(owner: str, repo: str) -> str:
    """
    Check project dependencies against the OSV (Open Source Vulnerabilities) database.
    Calls get_dependencies internally, then queries osv.dev for known CVEs.

    Args:
        owner: Repository owner
        repo: Repository name

    Returns:
        JSON string with list of vulnerable packages and their CVE details
    """
    def get_vuln_details(vuln_id: str) -> dict:
        response = requests.get(f"{OSV_API}/vulns/{vuln_id}")
        if response.status_code == 200:
            return response.json()
        return {"id": vuln_id, "error": "Details not found"}

    ecosystems = {}

    # Step 1: get dependencies
    deps = json.loads(get_dependencies(owner, repo))

    for d in deps["ecosystems"]:

        if "error" in d:
            return json.dumps({"error": d["error"]})

        if d["ecosystem"] == "python":
            ecosystem = "PyPI"
        elif d["ecosystem"] == "npm":
            ecosystem = "npm"
        elif d["ecosystem"] == "java":
            ecosystem = "Maven"
        else:
            ecosystem = "unknown"
        dependencies = d["dependencies"]

        # Step 2: build OSV batch query
        queries = []
        for dep in dependencies:
            query = {"package": {"name": dep["name"], "ecosystem": ecosystem}}
            if dep.get("version"):
                query["version"] = dep["version"]
            queries.append(query)

        if not queries:
            return json.dumps({"message": "No dependencies to check."})

        # OSV batch endpoint accepts up to 1000 queries
        osv_response = requests.post(
            f"{OSV_API}/querybatch",
            json={"queries": queries},
            headers={"Content-Type": "application/json"}
        )

        osv_response.raise_for_status()
        osv_results = osv_response.json().get("results", [])

        # Step 3: collect vulnerable packages
        vulnerabilities = []
        for dep, result in zip(dependencies, osv_results):
            vulns = result.get("vulns", [])
            if vulns:
                for v in vulns:
                    details = get_vuln_details(v["id"])
                    v.update(details)  # enrich with details for severity, summary, etc.
                    vulnerabilities.append({
                        "package": dep["name"],
                        "version": dep.get("version"),
                        "vulnerability_count": len(vulns),
                        "vulnerabilities": [
                            {
                                "id": v["id"],
                                "summary": v.get("summary") or v.get("details", "No summary available")[:200],
                                "severity": v.get("database_specific", {}).get("severity", "unknown"),
                                "fixed_in": [
                                    event["fixed"]
                                    for affected in v.get("affected", [])
                                    for rng in affected.get("ranges", [])
                                    if rng.get("type") == "ECOSYSTEM"
                                    for event in rng.get("events", [])
                                    if "fixed" in event
                                ]
                            }
                            for v in vulns[:5]  # cap at 5 CVEs per package
                        ]
                    })
        
        ecosystems.setdefault(
            ecosystem, {
                "dependencies": dependencies,
                "packages_checked": len(dependencies),
                "vulnerabilities": len(vulnerabilities),
                "results": vulnerabilities
            }
        )

    return json.dumps({
        "results": [
            {
                "ecosystem": eco,
                "packages_checked": data["packages_checked"],
                "vulnerabilities_found": data["vulnerabilities"],
                "vulnerable_packages": data["results"]
            }
            for eco, data in ecosystems.items()
        ]
    }, indent=2)

In [126]:
check_vulnerabilities("psf", "requests")

GHSA-8rrh-rw8j-w5fx Wheel Affected by Arbitrary File Permission Modification via Path Traversal in wheel unpack HIGH
GHSA-qwmp-2cf2-g9g6 pypa/wheel vulnerable to Regular Expression denial of service (ReDoS) HIGH
PYSEC-2022-43017 No summary unknown


'{\n  "results": [\n    {\n      "ecosystem": "PyPI",\n      "packages_checked": 24,\n      "vulnerabilities_found": 3,\n      "vulnerable_packages": [\n        {\n          "package": "wheel",\n          "version": null,\n          "vulnerability_count": 3,\n          "vulnerabilities": [\n            {\n              "id": "GHSA-8rrh-rw8j-w5fx",\n              "summary": "Wheel Affected by Arbitrary File Permission Modification via Path Traversal in wheel unpack",\n              "severity": "HIGH",\n              "fixed_in": [\n                "0.46.2"\n              ]\n            },\n            {\n              "id": "GHSA-qwmp-2cf2-g9g6",\n              "summary": "No summary available",\n              "severity": "unknown",\n              "fixed_in": []\n            },\n            {\n              "id": "PYSEC-2022-43017",\n              "summary": "No summary available",\n              "severity": "unknown",\n              "fixed_in": []\n            }\n          ]\n        },

## Tool Schema

Here are the schema of each tool which you will provide to the LLM.

In [13]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_repo_info",
            "description": "Fetch general metadata about a GitHub repository: language, license, stars, last activity, open issues, topics.",
            "parameters": {
                "type": "object",
                "properties": {
                    "owner": {"type": "string", "description": "Repository owner (user or organization)"},
                    "repo": {"type": "string", "description": "Repository name"}
                },
                "required": ["owner", "repo"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_file_tree",
            "description": "Fetch the full file and directory structure of a repository. Returns structural signals like presence of tests, CI config, Docker, README, and dependency files.",
            "parameters": {
                "type": "object",
                "properties": {
                    "owner": {"type": "string", "description": "Repository owner"},
                    "repo": {"type": "string", "description": "Repository name"},
                    "max_files": {"type": "integer", "description": "Max number of paths to return (default: 100)", "default": 100}
                },
                "required": ["owner", "repo"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_file_content",
            "description": "Read the content of a specific file in the repository. Use this to inspect README, configuration files, or source code.",
            "parameters": {
                "type": "object",
                "properties": {
                    "owner": {"type": "string", "description": "Repository owner"},
                    "repo": {"type": "string", "description": "Repository name"},
                    "path": {"type": "string", "description": "File path relative to repo root (e.g. 'README.md', 'src/app.py')"}
                },
                "required": ["owner", "repo", "path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_dependencies",
            "description": "Extract the list of dependencies from the project's dependency file (requirements.txt, package.json, pyproject.toml). Detects ecosystem automatically.",
            "parameters": {
                "type": "object",
                "properties": {
                    "owner": {"type": "string", "description": "Repository owner"},
                    "repo": {"type": "string", "description": "Repository name"}
                },
                "required": ["owner", "repo"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_vulnerabilities",
            "description": "Check the repository's dependencies against the OSV (Open Source Vulnerabilities) database to find known CVEs. Returns vulnerable packages with severity and fix versions. Calls get_dependencies internally, so no need to call it separately.",
            "parameters": {
                "type": "object",
                "properties": {
                    "owner": {"type": "string", "description": "Repository owner"},
                    "repo": {"type": "string", "description": "Repository name"}
                },
                "required": ["owner", "repo"]
            }
        }
    }
]

## Tool Mapping

This code handles tool mapping and execution.

In [14]:
mapping_tool_function = {
    "get_repo_info": get_repo_info,
    "get_file_tree": get_file_tree
}

def execute_tool(tool_name, tool_args):
    if isinstance(tool_args, str):
        tool_args = json.loads(tool_args)

    result = mapping_tool_function[tool_name](**tool_args)

    if result is None:
        result = "The operation completed but didn't return any results."

    elif isinstance(result, list):
        result = ', '.join(result)

    elif isinstance(result, dict):
        # Convert dictionaries to formatted JSON strings
        result = json.dumps(result, indent=2)

    else:
        # For any other type, convert using str()
        result = str(result)
    return result

## Chatbot Code

The chatbot handles the user's queries one by one, but it does not persist memory across the queries.

In [32]:
client = openai.OpenAI()

history = [
    {
        "role": "system",
        "content": (
            "You are a GitHub repository auditor. "
            "When the user mentions a repository, always identify the owner and repo name. "
            "Repositories are usually written as 'owner/repo' (e.g. 'vercel/next.js'). "
            "If the user provides them separately or informally, infer both before calling any tool. "
            "If you cannot determine the owner, ask the user before proceeding."
        )
    }
]

### Query Processing

In [33]:
def process_query(query):
    global history
    
    history.append({'role': 'user', 'content': query})
    
    response = client.chat.completions.create(
        model = 'gpt-4o-mini',
        messages = history,
        tools = tools,
        max_completion_tokens = 500
    )

    process_query_flag = True
    while process_query_flag:

        message = response.choices[0].message

        if not message.tool_calls:
            print(message.content)
            history.append({'role': 'assistant', 'content': message.content})
            process_query_flag = False
            continue

        history.append({
            'role': 'assistant',
            'content': message.content or "",
            "tool_calls": message.tool_calls
        })

        for tool_call in message.tool_calls:

            tool_id = tool_call.id
            tool_args = json.loads(tool_call.function.arguments)
            tool_name = tool_call.function.name

            print(f"Calling tool {tool_name}({tool_args})")
            result = execute_tool(tool_name, tool_args)

            history.append({
                "role": "tool",
                "tool_call_id": tool_id,
                "content": json.dumps(result),
            })

        response = client.chat.completions.create(
            model = 'gpt-4o-mini',
            messages = history,
            tools = tools,
            max_completion_tokens = 500
        )

### Chat Loop

In [34]:
def chat_loop():
    print("Type your queries or 'quit' to exit.")
    while True:
        try:
            query = input("\nQuery: ").strip()
            if query.lower() == 'quit':
                break

            process_query(query)
            print("\n")
        except Exception as e:
            print(f"\nError: {str(e)}")

Feel free to interact with the auditor. Here are some example queries:

- `Audit the repository tiangolo/fastapi`
- `Does psf/requests have any known vulnerabilities?`
- `What's the structure of vercel/next.js? Does it have tests and CI?`
- `Check the dependencies of encode/httpx and tell me if any are outdated or vulnerable`

In [35]:
chat_loop()

Type your queries or 'quit' to exit.
¡Hola Simón! Soy un auditor de repositorios de GitHub y puedo ayudarte a obtener información sobre repositorios, como su metadata (lenguaje, licencia, estrellas, etc.) y la estructura de archivos y directorios. Si tienes un repositorio en mente, simplemente menciona su nombre y te proporcionaré la información que necesites.


Sí, tu nombre es Simón. ¿Hay algo específico en lo que te gustaría que te ayude con respecto a un repositorio de GitHub?


